# GITS Scanner — Apple PoC

**Purpose:** Validate whether the revenue-weighted Google Trends index leads Apple's quarterly segment revenue and stock price.

**Decision rule:** Pearson r > 0.5 at lead 1-2 quarters → continue; otherwise stop and reconsider the thesis.

**Prerequisites:**
1. `reference/apple_revenue_weights.csv` filled out (manually from 10-Q)
2. `python scripts/fetch_trends.py` executed
3. `python scripts/fetch_prices.py` executed

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
from gits.config import WEIGHTS_CSV
from gits.storage.duckdb_io import get_conn, init_schema, read_trends, read_prices, read_quarterly_revenue
from gits.engine.normalize import pivot_trends_wide, align_to_apple_fiscal_quarters
from gits.engine.weighting import load_weights_from_csv, compute_gits_index
from gits.analysis.backtest import lead_lag_correlation, yoy_change
from gits.analysis.plots import three_axis_chart, segment_contribution_chart, lead_lag_chart

pd.set_option('display.max_rows', 30)

## 1. Load all data

In [ ]:
conn = get_conn()
init_schema(conn)

trends_long = read_trends(conn, geo='WW')
prices = read_prices(conn, ticker='AAPL').set_index('date')
qrev = read_quarterly_revenue(conn, ticker='AAPL').set_index('quarter_end')
weights_long = load_weights_from_csv(WEIGHTS_CSV)

print(f'Trends: {len(trends_long)} rows, segments={trends_long["segment"].unique().tolist()}')
print(f'Prices: {len(prices)} days, {prices.index.min()} → {prices.index.max()}')
print(f'Quarterly rev: {len(qrev)} quarters')
print(f'Segment weights: {weights_long["quarter_end"].nunique()} fiscal quarters loaded')
weights_long.head()

## 2. Compute GITS Index

In [ ]:
traffic_wide = pivot_trends_wide(trends_long)
gits = compute_gits_index(traffic_wide, weights_long)
gits.tail(10)

In [ ]:
fig = segment_contribution_chart(gits, title='Apple — Weighted Segment Contribution to GITS')
fig.show()

## 3. Three-axis overlay: GITS vs Revenue vs Stock

In [ ]:
gits_q = gits['gits'].resample('QS').mean()
fig = three_axis_chart(
    gits=gits_q,
    revenue=qrev['total_revenue_usd_m'],
    price=prices['adj_close'].resample('W').last(),
    title='Apple — GITS Index vs Quarterly Revenue vs AAPL Price',
)
fig.show()

## 4. Lead-lag correlation — THE PoC verdict

A positive `lead` value means GITS leads the target by N quarters.

In [ ]:
# GITS vs total revenue YoY
rev_yoy = yoy_change(qrev['total_revenue_usd_m'].sort_index(), periods=4)
gits_yoy = yoy_change(gits_q.dropna(), periods=4)

corr_rev = lead_lag_correlation(gits_yoy, rev_yoy, max_lead=4, freq='Q')
print('GITS YoY vs Total Revenue YoY:')
print(corr_rev)
lead_lag_chart(corr_rev, title='GITS YoY → Apple Revenue YoY').show()

In [ ]:
# GITS vs AAPL price (quarterly returns)
price_q = prices['adj_close'].resample('QS').last().pct_change()
corr_px = lead_lag_correlation(gits_yoy, price_q, max_lead=4, freq='Q')
print('GITS YoY vs AAPL Quarterly Return:')
print(corr_px)
lead_lag_chart(corr_px, title='GITS YoY → AAPL Quarterly Return').show()

## 5. PoC verdict

**Interpretation guide:**
- `lead = 0`: GITS this quarter correlates with revenue/price this quarter (coincident, not leading)
- `lead = 1, 2`: GITS this quarter correlates with revenue/price 1-2 quarters ahead (leading — what we want)
- `lead = -1, -2`: target leads GITS (search reflects past results — bad)

**Go criteria:** `pearson_r > 0.5` at `lead ∈ {1, 2}` AND `p_value < 0.05`.

Fill in your conclusion here once data is loaded.